In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

In [3]:
raw_dir = Path("data/raw")

pdf_files = list(raw_dir.glob("*.pdf"))
pdf_files

[WindowsPath('data/raw/Mission Saksham Anganwadi and Poshan 2.0 scheme guidelines.pdf'),
 WindowsPath('data/raw/Operational-Guidelines-of-PMAY-U-2.pdf'),
 WindowsPath('data/raw/PM-KMY - Operational Guidelines.pdf'),
 WindowsPath('data/raw/PMAJAY.pdf'),
 WindowsPath('data/raw/PMMVY.pdf'),
 WindowsPath('data/raw/RevisedPM-KISANOperationalGuidelines(English).pdf')]

In [35]:
import re

def clean_text(text:str) ->str:
    text = text.strip()
    text = re.sub(r"[^\S\n]+"," ",text)
    text = re.sub(r"\n{3,}","\n\n",text)
    text = re.sub(r"\.{2,}", "...", text)
    return text

In [40]:
def table_to_text(table_data) -> str:
    if not table_data or len(table_data) < 2:
        return ""

    headers = table_data[0]
    rows = []

    for row in table_data[1:]:
        fields = []

        for header, value in zip(headers, row):
            header = str(header).strip() if header else ""
            value = str(value).strip() if value else ""

            if header and value:
                fields.append(f"{header}: {value}")
            elif value:
                fields.append(value)

        if fields:
            rows.append("\n".join(fields))

    return "\n\n".join(rows)

In [44]:
def load_documents():
    documents = []

    pdf_files = sorted(raw_dir.glob("*.pdf"))

    for pdf_file in pdf_files:
        scheme_name = pdf_file.stem

        print(f"Processing: {pdf_file.name}")

        with fitz.open(pdf_file) as pdf:
            for page_number, page in enumerate(pdf, start=1):

                # Detect tables on the page
                tables = page.find_tables()

                # Case 1: Tables are present
                if tables.tables:

                    # Extract ordinary text
                    text = clean_text(page.get_text("text"))

                    if text:
                        documents.append({
                            "text": text,
                            "metadata": {
                                "source": pdf_file.name,
                                "page": page_number,
                                "scheme": scheme_name,
                                "content_type": "text"
                            }
                        })

                    # Extract each table separately
                    for table_index, table in enumerate(
                        tables.tables, start=1
                    ):
                        table_data = table.extract()

                        table_text = table_to_text(table_data)
                        table_text = clean_text(table_text)

                        if table_text:
                            documents.append({
                                "text": table_text,
                                "metadata": {
                                    "source": pdf_file.name,
                                    "page": page_number,
                                    "scheme": scheme_name,
                                    "content_type": "table",
                                    "table_index": table_index
                                }
                            })

                # Case 2: No tables are detected
                else:
                    text = clean_text(page.get_text("text"))

                    if text:
                        documents.append({
                            "text": text,
                            "metadata": {
                                "source": pdf_file.name,
                                "page": page_number,
                                "scheme": scheme_name,
                                "content_type": "text"
                            }
                        })

    print(f"Total records loaded: {len(documents)}")

    return documents

In [45]:
document_loaders = load_documents()

Processing: Mission Saksham Anganwadi and Poshan 2.0 scheme guidelines.pdf
Processing: Operational-Guidelines-of-PMAY-U-2.pdf
Processing: PM-KMY - Operational Guidelines.pdf
Processing: PMAJAY.pdf
Processing: PMMVY.pdf
Processing: RevisedPM-KISANOperationalGuidelines(English).pdf
Total records loaded: 409


In [38]:
len(document_loaders)

277

In [39]:
document_loaders

[{'text': 'Saksham Anganwadi and Poshan 2.0 \n \n \n \n \n \n \nMission Saksham \nAnganwadi and \nPoshan 2.0 \n \nScheme Guidelines \n \n \n \n \nMinistry of Women and Child Development \nGovernment of India',
  'metadata': {'source': 'Mission Saksham Anganwadi and Poshan 2.0 scheme guidelines.pdf',
   'page': 1,
   'scheme': 'Mission Saksham Anganwadi and Poshan 2.0 scheme guidelines'}},
 {'text': 'Saksham Anganwadi and Poshan 2.0 \n \n \nContents \n1. \nINTRODUCTION ... 3 \n1.1 \nVision ... 4 \n1.2 \nObjectives ... 4 \n1.3 \nImportant components of Saksham Anganwadi and Poshan 2.0 ... 5 \n2. \nINSTITUTIONAL FRAMEWORK FOR IMPLEMENTATION ... 7 \n2.1 \nFramework at Central Level ... 7 \n2.2 \nRoles and responsibilities of State Administration ... 8 \n2.3 \nRoles and responsibilities of District Administration ... 8 \n2.4 \nRoles and Responsibilities of Anganwadi Workers ... 9 \n2.5 \nRoles and responsibilities of Anganwadi Helpers ... 11 \n2.6 \nPoshan Tracker... 12 \n2.7 \nFund Flow ..